In [27]:
!pip install pandas-gbq google-auth --quiet

import os
import hashlib
import pandas as pd
import pandas_gbq
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_dark"

PROJECT_ID = "gdelt-494812"
DATASET_ID = "benin_2025"
KEY_PATH = "gdelt-494812-54aad2c4e931.json"


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [28]:
def get_credentials():
    if os.path.exists(KEY_PATH):
        from google.oauth2 import service_account
        return service_account.Credentials.from_service_account_file(KEY_PATH)
    return None

credentials = get_credentials()

In [29]:
os.makedirs("cache/", exist_ok=True)

def charger_donnees(query, force_reload=False):
    query_hash = hashlib.md5(query.encode()).hexdigest()[:8]
    cache_path = f"cache/cache_{query_hash}.csv"
    if os.path.exists(cache_path) and not force_reload:
        df = pd.read_csv(cache_path)
    else:
        df = pandas_gbq.read_gbq(query, project_id=PROJECT_ID, credentials=credentials)
        df.to_csv(cache_path, index=False)
    return df

In [30]:
query_lang = f"""
SELECT source_language, tone_category, COUNT(*) as total
FROM `{PROJECT_ID}.{DATASET_ID}.gkg_clean`
WHERE source_language IS NOT NULL
GROUP BY source_language, tone_category
ORDER BY total DESC
LIMIT 30
"""

df_lang = charger_donnees(query_lang)

fig = px.bar(
    df_lang,
    x="source_language",
    y="total",
    color="tone_category",
    barmode="stack",
    title="Sentiment par Langue Source des Articles au Bénin (2025)"
)
fig.update_layout(template="plotly_dark")
fig.show()

## 01 — L'anglais domine la narration d'un pays francophone

**~25 000 articles** en anglais contre **~12 000** en français — soit **2× plus** de couverture anglophone.

Le français béninois produit peu de contenu numérique indexé à l'échelle mondiale.
Le Bénin compte ~100 journaux et ~200 médias audiovisuels *(RSF, 2025)*, mais leur
diffusion reste locale. La francophonie béninoise parle à voix basse sur l'internet
mondial — et GDELT enregistre ce silence.

---

## 02 — Le sentiment négatif est structurel, pas conjoncturel

Dans chaque barre — `eng`, `fra`, `spa`, `por` — le **Négatif** et le **Très Négatif**
occupent la majeure partie de la hauteur, toutes langues confondues.

Ce pattern uniforme transcende les cultures éditoriales. En 2023–2025, le Bénin a connu
la suspension de médias indépendants et des restrictions du Code du numérique
*(Amnesty International, 2025)*. Le sentiment négatif dans les données est le reflet
d'une **presse sous tension**, pas une distorsion algorithmique.

---

## 03 — Neuf langues présentes, mais 50 invisibles

Les codes `eng`, `fra`, `spa`, `ita`, `por`, `rus`, `ell`, `bul`, `zho`, `ara`
représentent la quasi-totalité des articles.

Pourtant, le Bénin compte **plus de 50 langues vivantes** — fon, yoruba, bariba, dendi…
Ces langues, portées par la radio et la tradition orale, ne génèrent pas de texte
numérique indexable. GDELT ne capte pas la réalité vernaculaire du pays.

> **L'absence d'une langue dans les données n'est pas son silence — c'est son exclusion.**